In [12]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score


# ============================================================
# 1. LOAD YOUR CSV FILE
# ============================================================

csv_path = "Llama_Agreed.csv"
# csv_path = "GPT_Agreed.csv"
# csv_path = "CM_RR.csv"

df = pd.read_csv(csv_path)

print("Columns found:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())


# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "case_id",
    "field",
    "category",
    "human1",
    "human2",
    "agreement"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# 3. CLEAN HUMAN 1 / HUMAN 2 VALUES
# ============================================================

# Convert to numeric where possible
df["human1"] = pd.to_numeric(df["human1"], errors="coerce")
df["human2"] = pd.to_numeric(df["human2"], errors="coerce")

# Keep only rows where both coders have a valid decision
analysis_df = df.dropna(
    subset=["human1", "human2"]
).copy()

# Make sure decisions are binary
invalid_human1 = ~analysis_df["human1"].isin([0, 1])
invalid_human2 = ~analysis_df["human2"].isin([0, 1])

if invalid_human1.any():
    print("WARNING: Invalid Human 1 values:")
    print(analysis_df.loc[invalid_human1, "human1"].unique())

if invalid_human2.any():
    print("WARNING: Invalid Human 2 values:")
    print(analysis_df.loc[invalid_human2, "human2"].unique())

if invalid_human1.any() or invalid_human2.any():
    raise ValueError(
        "human1 and human2 must contain binary values: 0 or 1."
    )


# ============================================================
# 4. FUNCTION TO CALCULATE POOLED COHEN'S KAPPA
# ============================================================

def calculate_pooled_kappa(group):
    """
    Calculate pooled Cohen's kappa for all category-level
    binary decisions within a field.
    """

    human1 = group["human1"].astype(int)
    human2 = group["human2"].astype(int)

    # Observed agreement
    observed_agreement = (human1 == human2).mean()

    # Total number of decisions
    n = len(group)

    # Number of agreements
    agreements = (human1 == human2).sum()

    # Cohen's kappa
    #
    # If both coders have no variation, sklearn returns NaN.
    try:
        kappa = cohen_kappa_score(human1, human2)
    except Exception:
        kappa = np.nan

    # --------------------------------------------------------
    # Confusion matrix components
    # --------------------------------------------------------

    true_positive = ((human1 == 1) & (human2 == 1)).sum()
    true_negative = ((human1 == 0) & (human2 == 0)).sum()
    false_positive = ((human1 == 0) & (human2 == 1)).sum()
    false_negative = ((human1 == 1) & (human2 == 0)).sum()

    human1_positive = human1.sum()
    human2_positive = human2.sum()

    return pd.Series({
        "categories": group["category"].nunique(),
        "n_decisions": n,
        "human1_positive": human1_positive,
        "human2_positive": human2_positive,
        "agreements": agreements,
        "percent_agreement": observed_agreement * 100,

        "TP": true_positive,
        "TN": true_negative,
        "FP": false_positive,
        "FN": false_negative,

        "pooled_kappa": kappa
    })


# ============================================================
# 5. CALCULATE POOLED KAPPA FOR EACH FIELD
# ============================================================

pooled_results = (
    analysis_df
    .groupby("field", sort=False)
    .apply(calculate_pooled_kappa)
    .reset_index()
)


# ============================================================
# 6. ADD INTERPRETATION
# ============================================================

def interpret_kappa(kappa):

    if pd.isna(kappa):
        return "Not estimable"

    elif kappa < 0:
        return "Less than chance"

    elif kappa < 0.21:
        return "Slight"

    elif kappa < 0.41:
        return "Fair"

    elif kappa < 0.61:
        return "Moderate"

    elif kappa < 0.81:
        return "Substantial"

    else:
        return "Almost perfect"


pooled_results["interpretation"] = (
    pooled_results["pooled_kappa"]
    .apply(interpret_kappa)
)


# ============================================================
# 7. ROUND VALUES FOR DISPLAY
# ============================================================

pooled_results["percent_agreement"] = (
    pooled_results["percent_agreement"]
    .round(1)
)

pooled_results["pooled_kappa"] = (
    pooled_results["pooled_kappa"]
    .round(3)
)


# ============================================================
# 8. DISPLAY FINAL RESULTS
# ============================================================

final_table = pooled_results[
    [
        "field",
        "categories",
        "n_decisions",
        "human1_positive",
        "human2_positive",
        "percent_agreement",
        "pooled_kappa",
        "interpretation"
    ]
]

print("\n==============================================")
print("FIELD-LEVEL POOLED COHEN'S KAPPA")
print("==============================================\n")

print(
    final_table.to_string(index=False)
)


# ============================================================
# 9. SAVE RESULTS TO CSV
# ============================================================

# output_path = "pooled_kappa_results_CM_RR.csv"
# output_path = "pooled_kappa_results_LLM_Human.csv"
output_path = "pooled_kappa_results_Llama_Human.csv"

final_table.to_csv(
    output_path,
    index=False
)

print("\nResults saved to:")
print(output_path)

Columns found:
['case_id', 'field', 'category', 'human1', 'human2', 'agreement']

First 5 rows:
  case_id                                    field           category  human1  \
0  CS_001  attack_characteristics.attack_step_goal  availability_loss       0   
1  CS_002  attack_characteristics.attack_step_goal  availability_loss       0   
2  CS_003  attack_characteristics.attack_step_goal  availability_loss       0   
3  CS_004  attack_characteristics.attack_step_goal  availability_loss       0   
4  CS_005  attack_characteristics.attack_step_goal  availability_loss       1   

   human2  agreement  
0       0          1  
1       0          1  
2       0          1  
3       0          1  
4       1          1  

FIELD-LEVEL POOLED COHEN'S KAPPA

                                            field  categories  n_decisions  human1_positive  human2_positive  percent_agreement  pooled_kappa   interpretation
          attack_characteristics.attack_step_goal         9.0         90.0           

C:\Users\Nadeeka_92\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\Nadeeka_92\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\Nadeeka_92\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\Nadeeka_92\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\Nadeeka_92\AppData\Local\Temp\i

In [13]:
# ============================================================
# 8. DISPLAY FINAL RESULTS
# ============================================================

final_table = pooled_results[
    [
        "field",
        "categories",
        "n_decisions",
        "human1_positive",
        "human2_positive",
        "percent_agreement",
        "pooled_kappa",
        "interpretation"
    ]
].copy()


# Rename columns to make the output easier to read
display_table = final_table.rename(columns={
    "field": "Field",
    "categories": "Categories",
    "n_decisions": "N Decisions",
    "human1_positive": "Human 1 +",
    "human2_positive": "Human 2 +",
    "percent_agreement": "Agreement %",
    "pooled_kappa": "Cohen's κ",
    "interpretation": "Interpretation"
})


# Pretty-print table
from tabulate import tabulate

print("\n")
print("=" * 110)
print("                 FIELD-LEVEL POOLED COHEN'S KAPPA")
print("=" * 110)

print(
    tabulate(
        display_table,
        headers="keys",
        tablefmt="fancy_grid",
        showindex=False,
        floatfmt=".3f"
    )
)

print("=" * 110)


# ============================================================
# 9. SAVE RESULTS TO CSV
# ============================================================

# output_path = "pooled_kappa_results_CM_RR.csv"
# output_path = "pooled_kappa_results_LLM_Human.csv"
output_path = "pooled_kappa_results_Llama_Human.csv"

final_table.to_csv(
    output_path,
    index=False
)

print("\nResults saved to:")
print(output_path)





                 FIELD-LEVEL POOLED COHEN'S KAPPA
╒═══════════════════════════════════════════════════╤══════════════╤═══════════════╤═════════════╤═════════════╤═══════════════╤═════════════╤══════════════════╕
│ Field                                             │   Categories │   N Decisions │   Human 1 + │   Human 2 + │   Agreement % │   Cohen's κ │ Interpretation   │
╞═══════════════════════════════════════════════════╪══════════════╪═══════════════╪═════════════╪═════════════╪═══════════════╪═════════════╪══════════════════╡
│ attack_characteristics.attack_step_goal           │        9.000 │        90.000 │      16.000 │      20.000 │        80.000 │       0.377 │ Fair             │
├───────────────────────────────────────────────────┼──────────────┼───────────────┼─────────────┼─────────────┼───────────────┼─────────────┼──────────────────┤
│ attack_characteristics.attack_step                │       12.000 │       120.000 │      24.000 │      46.000 │        68.300 │       0.2